<a href="https://colab.research.google.com/github/danny-lin714/AI-Security-Learning-Path/blob/main/day01_micrograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 1：Autograd 的直覺（micrograd 前半）

**今天的範圍**：Karpathy《Neural Networks: Zero to Hero》第 1 集，從開頭看到「修正同一個節點被使用兩次的 bug」為止。拆解 tanh、和 PyTorch 對照、建 MLP 都留到 Day 2。

**使用方式**
1. 每一節先看對應的影片段落，再回來填 `TODO`。
2. 第 6 節的檢查 cell 不用改，全部 ✅ 就代表今天的產出完成。
3. 卡住超過 15 分鐘再回頭看影片或參考 Karpathy 的 micrograd repo，不要一開始就照抄。

**今天的產出**：這份能跑通的 notebook，commit 到你的 repo。

## 1. 單一輸入的導數
影片段落：*derivative of a simple function with one input*。重點是導數的定義：輸入往右推一點點，輸出會變多少。

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

def f(x):
    return 3*x**2 - 4*x + 5

xs = np.arange(-5, 5, 0.25)
plt.plot(xs, f(xs))
plt.title("f(x) = 3x^2 - 4x + 5")
plt.show()

In [ ]:
# TODO：用 (f(x + h) - f(x)) / h 估計 x = 3.0 的斜率，
# 再和手算的 f'(x) = 6x - 4 比較。試試看 h 越來越小時會發生什麼事（例如 1e-15）。
h = 0.0001
x = 3.0
slope = None  # TODO
print("數值估計:", slope, "  解析解:", 6*x - 4)

## 2. 多個輸入的導數
影片段落：*derivative of a function with multiple inputs*。

In [ ]:
# d = a*b + c。分別把 a、b、c 推一點點，估計 d 對它們的偏導數。
# 預期：dd/da = b = -3，dd/db = a = 2，dd/dc = 1
a, b, c = 2.0, -3.0, 10.0
h = 0.0001

d1 = a*b + c
# TODO：算 d2（某一個變數加上 h 之後的 d），印出 (d2 - d1) / h

## 3. Value 類別
影片段落：*starting the core Value object* 之後的幾段。影片會分好幾段慢慢長出這些欄位，跟著影片進度填就好，不用一次寫完：

- 先有 `data`、`_prev`、`_op`，能做加法和乘法、記住運算圖
- 手動 backprop 那段開始用到 `grad`
- 最後加上 `_backward` 和 `backward()`，讓梯度自動往回傳

這一版先假設 `other` 一定是 `Value`；支援 `Value + 1` 這種寫法是 Day 2 的事。

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0                # dL/d(這個節點)，一開始是 0
        self._backward = lambda: None  # 把 out.grad 傳給子節點的函式
        self._prev = set(_children)    # 這個節點是由哪些節點算出來的
        self._op = _op                 # 產生這個節點的運算
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        out = None  # TODO：Value(self.data + other.data, (self, other), '+')

        def _backward():
            pass    # TODO：加法的局部導數是多少？

        # TODO：把 _backward 掛到 out 上
        return out

    def __mul__(self, other):
        out = None  # TODO

        def _backward():
            pass    # TODO：乘法的局部導數是多少？

        # TODO：把 _backward 掛到 out 上
        return out

    def tanh(self):
        x = self.data
        t = None    # TODO：(e^{2x} - 1) / (e^{2x} + 1)
        out = None  # TODO

        def _backward():
            pass    # TODO：d tanh(x) / dx = 1 - tanh(x)^2

        # TODO：把 _backward 掛到 out 上
        return out

    def backward(self):
        # TODO：
        # 1. 用拓撲排序，把運算圖裡所有節點排成「子節點在父節點前面」的順序
        # 2. self.grad = 1.0
        # 3. 依反向順序，對每個節點呼叫 node._backward()
        pass

## 4. 視覺化運算圖（已寫好，直接執行）
需要 graphviz，Colab 已內建。這段不是今天的學習重點。

In [ ]:
from graphviz import Digraph

def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'})
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        dot.node(name=uid, shape='record',
                 label=f"{n.label} | data {n.data:.4f} | grad {n.grad:.4f}")
        if n._op:
            dot.node(name=uid + n._op, label=n._op)
            dot.edge(uid + n._op, uid)
    for child, parent in edges:
        dot.edge(str(id(child)), str(id(parent)) + parent._op)
    return dot

## 5. 手動 backprop
影片段落：*manual backpropagation example #1* 和 *#2*。先**手動**設定每個節點的 `grad`，用 chain rule 自己算，不要呼叫 `backward()`。算完用 `draw_dot` 看一遍。

In [ ]:
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
e = a * b;  e.label = 'e'
d = e + c;  d.label = 'd'
f = Value(-2.0, label='f')
L = d * f;  L.label = 'L'

# TODO：從 L 往回推，手動設定每個節點的 grad
# L.grad = 1.0
# f.grad = ...
# d.grad = ...

draw_dot(L)

In [ ]:
# 影片中的單一神經元：o = tanh(x1*w1 + x2*w2 + b)
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
b = Value(6.8813735870195432, label='b')

x1w1 = x1 * w1;  x1w1.label = 'x1*w1'
x2w2 = x2 * w2;  x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2;  x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b;  n.label = 'n'
o = n.tanh();  o.label = 'o'

# TODO：先手動填 grad；寫好 backward() 之後，改成 o.backward() 看結果是否一樣
draw_dot(o)

## 6. 檢查（不用改，直接執行）
寫完 `backward()` 後執行。每個檢查都重新建立運算圖，不會受前面手動設定的 grad 影響。

- 檢查 1、2：和影片裡的數字對照
- 檢查 3：拿 `backward()` 的結果和數值微分比對（gradient check）。這是驗證梯度實作最可靠的方法，Day 13 討論攻擊評估時會再遇到同樣的精神
- 檢查 4：專門抓影片後段討論的一個 bug；沒過的話回去看那段

In [ ]:
def check(name, got, expected, tol=1e-4):
    ok = abs(got - expected) < tol
    print(f"{'✅' if ok else '❌'} {name}: got {got:.4f}, expected {expected:.4f}")
    return ok

results = []

# 檢查 1：L = (a*b + c) * f
a, b, c, f = Value(2.0), Value(-3.0), Value(10.0), Value(-2.0)
L = (a*b + c) * f
L.backward()
print("檢查 1：L = (a*b + c) * f")
results += [check("L.data", L.data, -8.0),
            check("a.grad", a.grad, 6.0),
            check("b.grad", b.grad, -4.0),
            check("c.grad", c.grad, -2.0),
            check("f.grad", f.grad, 4.0)]

# 檢查 2：單一神經元
x1, x2 = Value(2.0), Value(0.0)
w1, w2 = Value(-3.0), Value(1.0)
bias = Value(6.8813735870195432)
o = (x1*w1 + x2*w2 + bias).tanh()
o.backward()
print("\n檢查 2：o = tanh(x1*w1 + x2*w2 + b)")
results += [check("o.data", o.data, 0.7071),
            check("x1.grad", x1.grad, -1.5),
            check("w1.grad", w1.grad, 1.0),
            check("x2.grad", x2.grad, 0.5),
            check("w2.grad", w2.grad, 0.0),
            check("b.grad", bias.grad, 0.5)]

# 檢查 3：backward() 的梯度 vs 數值梯度
def neuron_float(x1, x2, w1, w2, b):
    return math.tanh(x1*w1 + x2*w2 + b)

inputs = [2.0, 0.0, -3.0, 1.0, 6.8813735870195432]
values = [x1, x2, w1, w2, bias]
names = ["x1", "x2", "w1", "w2", "b"]
h = 1e-6
print("\n檢查 3：backward() 的梯度 vs 數值梯度")
for i, (name, v) in enumerate(zip(names, values)):
    bumped = inputs.copy()
    bumped[i] += h
    numerical = (neuron_float(*bumped) - neuron_float(*inputs)) / h
    results.append(check(name, v.grad, numerical))

# 檢查 4：同一個節點被用不只一次
print("\n檢查 4：同一個節點被用不只一次")
a = Value(3.0)
y = a + a
y.backward()
results.append(check("a + a 的 a.grad", a.grad, 2.0))

a, b = Value(-2.0), Value(3.0)
g = (a*b) * (a+b)
g.backward()
results += [check("(a*b)*(a+b) 的 a.grad", a.grad, -3.0),
            check("(a*b)*(a+b) 的 b.grad", b.grad, -8.0)]

print(f"\n通過 {sum(results)}/{len(results)} 項")

## 7. 加分題：輸入的梯度，就是對抗樣本的起點
檢查 2 的神經元裡，`x1.grad` 和 `x2.grad` 是「輸出對**輸入**的梯度」。訓練時我們沿著權重的梯度更新權重；攻擊時，權重固定不動，改沿著輸入的梯度去推輸入。Day 8 的 FGSM 就是這一行：

$$x' = x + \varepsilon \cdot \operatorname{sign}(\nabla_x L)$$

試試看：把 `x1`、`x2` 各往自己梯度的 sign 方向推 ε = 0.1，輸出 `o` 會往哪個方向變？變了多少？

In [ ]:
eps = 0.1
sign = lambda g: (g > 0) - (g < 0)

# TODO：用檢查 2 算出的 x1.grad、x2.grad 產生 x1_adv、x2_adv，
# 再用 neuron_float 算出新的輸出，和原本的 0.7071 比較（權重和 b 不變）
x1_adv = None  # TODO
x2_adv = None  # TODO

## 8. 收尾筆記
用自己的話回答下面三題，寫在這個 cell 裡（Day 2 的筆記會接著用）：

1. 為什麼 `backward()` 要從 `self.grad = 1.0` 開始？
2. 拓撲排序在保證什麼？如果順序錯了會怎樣？
3. 每個運算的 `_backward` 只需要知道哪些資訊？為什麼這樣就足以算出整張圖的梯度？

**我的回答：**